In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from meta_analytics.dataframes import GlucoseEndpointsByDate2


In [ ]:
cls = GlucoseEndpointsByDate2()
df = cls.df


In [ ]:
df.endpoint_label.value_counts()

In [ ]:
subjects_df = cls.working_df.copy()
subjects_df = subjects_df.drop(columns=["endpoint"])

In [ ]:
subjects_df = subjects_df.sort_values(by=["subject_identifier", "visit_datetime"]).reset_index(drop=True).drop_duplicates(subset=["subject_identifier"], keep="last")
df_with_endpoint = subjects_df.merge(df[["subject_identifier", "endpoint", "endpoint_label"]], on="subject_identifier", how="left").sort_values("subject_identifier").reset_index(drop=True)
df_with_endpoint.loc[df_with_endpoint.endpoint.isna(), "endpoint"] = 0

In [ ]:
df_with_endpoint["days_to_event"] = (df_with_endpoint["visit_datetime"] - df_with_endpoint["baseline_datetime"]).dt.days
df_with_endpoint["person_years"] =  ((df_with_endpoint["visit_datetime"] - df_with_endpoint["baseline_datetime"]).dt.days)/365.25
df_with_endpoint['log_time'] = np.log(df_with_endpoint['person_years'])


In [ ]:
random_values = np.random.choice(["A", "B"], size=len(df_with_endpoint))
df_with_endpoint['group'] = random_values


In [ ]:
df_with_endpoint

In [ ]:
# Fit the Poisson model
# 'status' is your count (0 or 1), 'group' is your study arm
formula = "endpoint ~ group"
model = smf.poisson(formula, data=df_with_endpoint, offset=df_with_endpoint['log_time']).fit()

print(model.summary())

In [ ]:
model.params

In [ ]:
# Incidence Rate Ratio (IRR)
print("IRR", np.exp(model.params['group[T.B]']), "CI", np.exp(model.conf_int().loc['group[T.B]', 0]),"-", np.exp(model.conf_int().loc['group[T.B]', 1]))
# Lower 95% CI
# print(np.exp(model.conf_int().loc['group[T.B]', 0]))
# Upper 95% CI
# print(np.exp(model.conf_int().loc['group[T.B]', 1]))